## F1 Race Strategy Notebook

This notebook follows one explainable decision flow: raw race signals -> risk markers -> recommended action.

In [7]:
import pandas as pd
from pathlib import Path

sample_path = Path('../data/sample_strategy_cases.csv')
df = pd.read_csv(sample_path)
df.head()

,lap,position,tyre_age,tyre_compound,gap_to_car_ahead,attack_window,energy_available,overtake_mode,safety_car_probability
0,24,4,12.0,Soft,0.7,True,76,True,0.14
1,27,3,18.0,Medium,0.4,True,62,True,0.08
2,31,5,23.0,Hard,1.2,False,44,False,0.20
3,29,6,16.0,Soft,1.0,True,80,True,0.12
4,33,2,25.0,Medium,0.9,False,58,False,0.25


In [8]:
import numpy as np

df['attack_score'] = np.where(df['attack_window'] & (df['energy_available'] > 50), 1, 0)
df['risk_score'] = df['safety_car_probability'] + np.where(df['tyre_age'] > 20, 0.10, 0.02)

risk_summary = df.groupby('risk_score').agg({
    'lap': 'count',
    'energy_available': 'mean',
    'attack_score': 'sum'
})

risk_summary

,lap,energy_available,attack_score
risk_score,,,
0.10,1,62.0,1
0.14,1,80.0,1
0.16,1,76.0,1
0.30,1,44.0,0
0.35,1,58.0,0


## Explainable Strategy Flow

1. Identify tyre age and track position.
2. Check whether the car is in an attack window.
3. Estimate energy available for overtake mode.
4. Attach risk based on safety car and tyre uncertainty.
5. Recommend the fastest explainable route.

In [9]:
def recommend_strategy(row):
    if row['attack_window'] and row['overtake_mode'] and row['energy_available'] > 50:
        return 'Attack / Overtake Mode'
    if row['tyre_age'] > 20:
        return 'Pit now'
    return 'Stay out and monitor'

sample_recommendations = df.copy()
sample_recommendations['strategy_recommendation'] = sample_recommendations.apply(recommend_strategy, axis=1)
sample_recommendations[['lap', 'position', 'tyre_compound', 'strategy_recommendation', 'risk_score']]

,lap,position,tyre_compound,strategy_recommendation,risk_score
0,24,4,Soft,Attack / Overtake Mode,0.16
1,27,3,Medium,Attack / Overtake Mode,0.10
2,31,5,Hard,Pit now,0.30
3,29,6,Soft,Attack / Overtake Mode,0.14
4,33,2,Medium,Pit now,0.35


## Project Narrative

This project follows one clear story:

Raw race signals -> strategy decisions -> risk explanation -> recommended action.

It is not a generic winner predictor. It is a decision-intelligence project that asks:

What should the team do next, and what is the downside risk of that decision?